# StockSage — Exploratory Data Analysis

Prototyping notebook for the StockSage pipeline. Use this to sanity-check raw
downloads, inspect the engineered indicators, and eyeball model inputs before
committing changes to `src/`.

**Workflow**
1. `python -m src.data_loader` to populate `data/raw/`.
2. `python -m src.preprocessing` to populate `data/processed/` and fit scalers.
3. Explore below.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # make `src` importable from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
TICKER = "AAPL"

## 1. Raw OHLCV

In [ ]:
raw = pd.read_csv(f"../data/raw/{TICKER}.csv", parse_dates=["Date"])
print(raw.shape)
raw.tail()

In [ ]:
raw.set_index("Date")["Close"].plot(figsize=(12, 4), title=f"{TICKER} adjusted close")
plt.tight_layout()

## 2. Engineered indicators

Reuse the exact feature logic that the training pipeline uses, so the notebook
never drifts from production.

In [ ]:
from src.preprocessing import add_indicators, FEATURE_COLS

feat = add_indicators(raw.copy()).dropna().reset_index(drop=True)
print("Features:", FEATURE_COLS)
feat[["Date", "Close", *FEATURE_COLS]].tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
feat.set_index("Date")[["Close", "SMA_10", "SMA_50"]].plot(ax=axes[0], title="Trend")
feat.set_index("Date")[["MACD", "MACD_signal"]].plot(ax=axes[1], title="MACD")
feat.set_index("Date")[["RSI_14"]].plot(ax=axes[2], title="RSI")
axes[2].axhline(70, ls="--", c="r"); axes[2].axhline(30, ls="--", c="g")
plt.tight_layout()

## 3. Next steps

- Train: `python src/train.py --ticker AAPL --seq_len 60 --epochs 100`
- Inspect `models/saved/AAPL_metrics.json` for the held-out MAE.
- Launch the dashboard: `python app/dashboard.py`